<a href="https://colab.research.google.com/github/Ajendra11/Concepts-and-Technologies-of-AI/blob/main/AjendraRai_Worksheet8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:

import numpy as np
import pandas as pd
from sklearn.datasets import load_iris, load_wine
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, f1_score, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

In [6]:
# ============================================================================
# PART 1: Custom Decision Tree Implementation
# ============================================================================

class CustomDecisionTree:
    def __init__(self, max_depth=None):
        self.max_depth = max_depth
        self.tree = None

    def fit(self, X, y):
        self.tree = self._build_tree(X, y)

    def _build_tree(self, X, y, depth=0):
        num_samples, num_features = X.shape
        unique_classes = np.unique(y)

        if len(unique_classes) == 1:
            return {'class': unique_classes[0]}
        if num_samples == 0 or (self.max_depth and depth >= self.max_depth):
            return {'class': np.bincount(y).argmax()}

        best_info_gain = -float('inf')
        best_split = None

        for feature_idx in range(num_features):
            thresholds = np.unique(X[:, feature_idx])
            for threshold in thresholds:
                left_mask = X[:, feature_idx] <= threshold
                right_mask = ~left_mask
                left_y = y[left_mask]
                right_y = y[right_mask]

                if len(left_y) == 0 or len(right_y) == 0:
                    continue

                info_gain = self._information_gain(y, left_y, right_y)

                if info_gain > best_info_gain:
                    best_info_gain = info_gain
                    best_split = {
                        'feature_idx': feature_idx,
                        'threshold': threshold,
                        'left_mask': left_mask,
                        'right_mask': right_mask,
                    }

        if best_split is None:
            return {'class': np.bincount(y).argmax()}

        left_tree = self._build_tree(X[best_split['left_mask']],
                                     y[best_split['left_mask']], depth + 1)
        right_tree = self._build_tree(X[best_split['right_mask']],
                                      y[best_split['right_mask']], depth + 1)

        return {
            'feature_idx': best_split['feature_idx'],
            'threshold': best_split['threshold'],
            'left_tree': left_tree,
            'right_tree': right_tree
        }

    def _information_gain(self, parent, left, right):
        parent_entropy = self._entropy(parent)
        left_entropy = self._entropy(left)
        right_entropy = self._entropy(right)
        weighted_avg = (len(left) / len(parent)) * left_entropy + (len(right) / len(parent)) * right_entropy
        return parent_entropy - weighted_avg

    def _entropy(self, y):
        class_probs = np.bincount(y) / len(y)
        return -np.sum(class_probs * np.log2(class_probs + 1e-9))

    def predict(self, X):
        return [self._predict_single(x, self.tree) for x in X]

    def _predict_single(self, x, tree):
        if 'class' in tree:
            return tree['class']
        feature_val = x[tree['feature_idx']]
        if feature_val <= tree['threshold']:
            return self._predict_single(x, tree['left_tree'])
        else:
            return self._predict_single(x, tree['right_tree'])

# Load Iris dataset
data = load_iris()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Custom Decision Tree
custom_tree = CustomDecisionTree(max_depth=3)
custom_tree.fit(X_train, y_train)
y_pred_custom = custom_tree.predict(X_test)
accuracy_custom = accuracy_score(y_test, y_pred_custom)

# Train Scikit-learn Decision Tree
sklearn_tree = DecisionTreeClassifier(max_depth=3, random_state=42)
sklearn_tree.fit(X_train, y_train)
y_pred_sklearn = sklearn_tree.predict(X_test)
accuracy_sklearn = accuracy_score(y_test, y_pred_sklearn)

print("PART 1 - Iris Dataset:")
print(f"Custom Decision Tree Accuracy: {accuracy_custom:.4f}")
print(f"Scikit-learn Decision Tree Accuracy: {accuracy_sklearn:.4f}\n")

PART 1 - Iris Dataset:
Custom Decision Tree Accuracy: 1.0000
Scikit-learn Decision Tree Accuracy: 1.0000



In [7]:
# ============================================================================
# PART 2: Wine Dataset - Classification
# ============================================================================

# Load Wine dataset
wine_data = load_wine()
X_wine, y_wine = wine_data.data, wine_data.target
X_train_wine, X_test_wine, y_train_wine, y_test_wine = train_test_split(
    X_wine, y_wine, test_size=0.2, random_state=42
)

# Train Decision Tree Classifier
dt_classifier = DecisionTreeClassifier(random_state=42)
dt_classifier.fit(X_train_wine, y_train_wine)
y_pred_dt = dt_classifier.predict(X_test_wine)
f1_dt = f1_score(y_test_wine, y_pred_dt, average='weighted')

# Train Random Forest Classifier
rf_classifier = RandomForestClassifier(random_state=42)
rf_classifier.fit(X_train_wine, y_train_wine)
y_pred_rf = rf_classifier.predict(X_test_wine)
f1_rf = f1_score(y_test_wine, y_pred_rf, average='weighted')

print("PART 2 - Wine Classification:")
print(f"Decision Tree F1 Score: {f1_dt:.4f}")
print(f"Random Forest F1 Score: {f1_rf:.4f}\n")

PART 2 - Wine Classification:
Decision Tree F1 Score: 0.9440
Random Forest F1 Score: 1.0000



In [4]:
# ============================================================================
# PART 3: Hyperparameter Tuning - GridSearchCV
# ============================================================================

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1
)

grid_search.fit(X_train_wine, y_train_wine)
y_pred_grid = grid_search.best_estimator_.predict(X_test_wine)
f1_grid = f1_score(y_test_wine, y_pred_grid, average='weighted')

print("PART 3 - GridSearchCV Results:")
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best CV F1 Score: {grid_search.best_score_:.4f}")
print(f"Test F1 Score: {f1_grid:.4f}\n")


PART 3 - GridSearchCV Results:
Best Parameters: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}
Best CV F1 Score: 0.9783
Test F1 Score: 1.0000



In [8]:
# ============================================================================
# PART 4: Wine Dataset - Regression
# ============================================================================

# Use alcohol content as target
X_wine_reg = np.delete(wine_data.data, 0, axis=1)
y_wine_reg = wine_data.data[:, 0]
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_wine_reg, y_wine_reg, test_size=0.2, random_state=42
)

# Train Decision Tree Regressor
dt_regressor = DecisionTreeRegressor(random_state=42)
dt_regressor.fit(X_train_reg, y_train_reg)
y_pred_dt_reg = dt_regressor.predict(X_test_reg)
r2_dt = r2_score(y_test_reg, y_pred_dt_reg)

# Train Random Forest Regressor
rf_regressor = RandomForestRegressor(random_state=42)
rf_regressor.fit(X_train_reg, y_train_reg)
y_pred_rf_reg = rf_regressor.predict(X_test_reg)
r2_rf = r2_score(y_test_reg, y_pred_rf_reg)

print("PART 4 - Wine Regression:")
print(f"Decision Tree R² Score: {r2_dt:.4f}")
print(f"Random Forest R² Score: {r2_rf:.4f}\n")

PART 4 - Wine Regression:
Decision Tree R² Score: 0.4775
Random Forest R² Score: 0.7416



In [9]:
# ============================================================================
# PART 5: Hyperparameter Tuning - RandomizedSearchCV
# ============================================================================

param_distributions = {
    'n_estimators': [50, 100, 150, 200, 250, 300],
    'max_depth': [None, 5, 10, 15, 20, 25, 30],
    'min_samples_split': [2, 5, 10, 15, 20]
}

random_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=42),
    param_distributions=param_distributions,
    n_iter=20,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    random_state=42
)

random_search.fit(X_train_reg, y_train_reg)
y_pred_random = random_search.best_estimator_.predict(X_test_reg)
r2_random = r2_score(y_test_reg, y_pred_random)

print("PART 5 - RandomizedSearchCV Results:")
print(f"Best Parameters: {random_search.best_params_}")
print(f"Best CV R² Score: {random_search.best_score_:.4f}")
print(f"Test R² Score: {r2_random:.4f}")

PART 5 - RandomizedSearchCV Results:
Best Parameters: {'n_estimators': 50, 'min_samples_split': 20, 'max_depth': 10}
Best CV R² Score: 0.5190
Test R² Score: 0.7588
